# BioCLIP-2 Species Identification on Local Images

This notebook loads **BioCLIP-2** and runs zero-shot **species identification** on a local image folder:

- `/Volumes/BigMacX/ebio/Pictures`

It mirrors the flow of [02_megadetector_local_images.ipynb](02_megadetector_local_images.ipynb): define paths, set up the
engine, discover images, run inference, then summarize and visualize results.

**Engine:** [`pybioclip`](https://github.com/Imageomics/pybioclip) is the official inference wrapper for BioCLIP; its
default model is `hf-hub:imageomics/bioclip-2` — the same weights published from the training/eval source under
`third-party/bioclip-2`. The `TreeOfLifeClassifier` predicts the full taxonomy (kingdom → species) using the
TreeOfLife-200M label embeddings.

It is safe to run even when the image folder is currently empty.

## Step 1 — Define paths

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
bioclip_root = repo_root / "third-party" / "bioclip-2"
image_dir = Path("/Volumes/BigMacX/ebio/Pictures")
output_dir = Path("/Volumes/BigMacX/ebio/project-id/outputs/bioclip-2")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repo_root}")
print(f"BioCLIP-2 source: {bioclip_root} (exists: {bioclip_root.exists()})")
print(f"Image directory: {image_dir}")
print(f"Output directory: {output_dir}")

## Step 2 — Install / verify the inference engine (pybioclip)

`pybioclip` pulls in `open_clip_torch`, `timm`, and `huggingface-hub`. The local `third-party/bioclip-2`
checkout is the training/evaluation source; here we use `pybioclip` for prediction against the same BioCLIP-2 weights.

In [ ]:
import importlib
import subprocess
import sys


def ensure_package(import_name, pip_name=None):
    try:
        importlib.import_module(import_name)
        print(f"'{import_name}' already available.")
    except ImportError:
        pip_name = pip_name or import_name
        print(f"Installing '{pip_name}' ...")
        subprocess.run([sys.executable, "-m", "pip", "install", pip_name], check=True)
        importlib.import_module(import_name)
        print(f"Installed '{pip_name}'.")


ensure_package("bioclip", "pybioclip")

## Step 3 — Smoke-test imports and select a device

CPU is used by default for stability on macOS. Set `device = "mps"` to try Apple GPU acceleration.

In [ ]:
import torch
from bioclip import TreeOfLifeClassifier, Rank, BIOCLIP_MODEL_STR

print(f"torch: {torch.__version__}")
print(f"Default BioCLIP model: {BIOCLIP_MODEL_STR}")

device = "cpu"  # options: "cpu", "mps", "cuda"
if device == "mps" and not torch.backends.mps.is_available():
    print("MPS not available; falling back to CPU.")
    device = "cpu"
print(f"Using device: {device}")

## Step 4 — Check image folder contents

In [ ]:
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
images = sorted([p for p in image_dir.glob("**/*") if p.is_file() and p.suffix.lower() in image_exts])

print(f"Found {len(images)} image(s)")
for p in images[:20]:
    print(" -", p)

if not images:
    print("\nNo images found yet. Populate the folder and rerun this notebook.")

## Step 5 — Load BioCLIP-2 (Tree-of-Life classifier)

> The **first** run downloads the BioCLIP-2 weights and the TreeOfLife-200M label embeddings from Hugging Face
> (a few GB). These are cached under `~/.cache/huggingface` for subsequent runs, so later loads are fast.

In [ ]:
classifier = TreeOfLifeClassifier(device=device)
print(f"Loaded model: {classifier.model_str}")

## Step 6 — Run species identification

Predicts the top-`k` species per image and writes a tidy CSV of all predictions to the output folder.

In [ ]:
import pandas as pd

# --- Options ---------------------------------------------------------------
rank = Rank.SPECIES   # predict down to species (also: GENUS, FAMILY, ORDER, ...)
top_k = 5             # number of top predictions kept per image
# ---------------------------------------------------------------------------

if not images:
    raise RuntimeError("No images available. Add images to the folder and rerun Step 4.")

image_paths = [str(p) for p in images]


def progress(done, total):
    print(f"  processed {done}/{total}", end="\r")


predictions = classifier.predict(image_paths, rank=rank, k=top_k, callback=progress)
print()  # newline after progress bar

df = pd.DataFrame(predictions)
results_csv = output_dir / "bioclip2_species_predictions.csv"
df.to_csv(results_csv, index=False)
print(f"Wrote {len(df)} prediction row(s) for {len(image_paths)} image(s) to:")
print(f"  {results_csv}")
df.head(top_k * 2)

## Step 7 — Top prediction per image (summary)

Keeps the single highest-scoring species per image and saves a compact summary CSV.

In [ ]:
preferred_cols = ["file_name", "kingdom", "phylum", "class", "order",
                  "family", "genus", "species", "common_name", "score"]

top1 = (
    df.sort_values("score", ascending=False)
      .groupby("file_name", as_index=False)
      .first()
)

summary_cols = [c for c in preferred_cols if c in top1.columns]
summary = top1[summary_cols].sort_values("file_name").reset_index(drop=True)

summary_csv = output_dir / "bioclip2_top1_summary.csv"
summary.to_csv(summary_csv, index=False)
print(f"Top-1 identification for {len(summary)} image(s). Saved to:")
print(f"  {summary_csv}")
summary

## Step 8 — Visualize identifications

Displays the first `max_preview` images inline, captioned with the predicted species, common name, and score.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage

max_preview = 6
preview = summary.head(max_preview)

if preview.empty:
    print("Nothing to visualize yet. Run Steps 4–6 on a folder that contains images.")

for _, row in preview.iterrows():
    path = row["file_name"]
    try:
        img = PILImage.open(path)
    except Exception as exc:
        print(f"  (skipped, cannot open) {path}: {exc}")
        continue

    species = row.get("species", "?")
    common = row.get("common_name", "")
    title = f"{species}" + (f" ({common})" if common else "") + f"\nscore = {row['score']:.3f}"

    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(title)
    plt.show()

## Optional — Classify against your own candidate labels

When you already know the plausible species/classes for a project, `CustomLabelsClassifier` restricts BioCLIP-2
to your list and returns a probability per label. Edit `candidate_labels`, then run the cell.

In [ ]:
from bioclip import CustomLabelsClassifier

candidate_labels = [
    "Odocoileus hemionus",   # mule deer
    "Puma concolor",         # mountain lion
    "Canis latrans",         # coyote
    "Lynx rufus",            # bobcat
    "Ursus americanus",      # black bear
]

if images:
    custom = CustomLabelsClassifier(candidate_labels, device=device)
    custom_preds = custom.predict([str(p) for p in images[:5]], k=len(candidate_labels))
    import pandas as pd
    display(pd.DataFrame(custom_preds))
else:
    print("Add images and rerun Step 4 to try custom-label classification.")

## Next steps

- Pair this with [02_megadetector_local_images.ipynb](02_megadetector_local_images.ipynb): run MegaDetector first to
  locate animals, crop each detection bounding box, then feed the crops here for species ID (BioCLIP-2 works best on
  tight subject crops).
- Change `rank` in Step 6 (e.g. `Rank.GENUS`) for coarser predictions, or raise `top_k` to inspect alternatives.
- Results are saved under `/Volumes/BigMacX/ebio/project-id/outputs/bioclip-2`.